# 🥁 Drum Copy — Google Colab 実行ノートブック

楽曲ファイル（wav / mp3）からドラムパートを分離し、MIDIファイルとして出力します。

```
入力音声 → [Demucs: stem分離] → ドラムwav → [omnizart: 自動採譜] → MIDI
```

## 実行前の準備

1. **GPUランタイムを有効化**  
   「ランタイム」→「ランタイムのタイプを変更」→「ハードウェアアクセラレータ: GPU（T4）」

2. **Google Drive にプロジェクトを配置**  
   `マイドライブ/drum-copy/` に本プロジェクトのフォルダをアップロード

3. **入力音声を配置**  
   `マイドライブ/drum-copy/input/` に処理したい wav / mp3 ファイルを入れる

4. **上から順にセルを実行**（「ランタイム」→「すべてのセルを実行」でも可）

> 出力 MIDI は `マイドライブ/drum-copy/output/` に保存されます。

## 1. GPU 確認

In [1]:
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"✅ GPU: {props.name}  ({props.total_memory / 1e9:.1f} GB VRAM)")
else:
    print("⚠️  GPU が検出されません。")
    print("「ランタイム」→「ランタイムのタイプを変更」→「ハードウェアアクセラレータ: GPU」を設定してください。")
    print("CPU でも実行できますが、処理に数十分かかる場合があります。")

✅ GPU: Tesla T4  (15.6 GB VRAM)


## 2. パッケージインストール

> ランタイムが変わるたびに再実行が必要です。初回は数分かかります。

In [3]:
# omnizart は vamp に依存しているが Colab 環境では不要なためパッチインストール
import subprocess, sys

def pip(args: str) -> None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args.split(), check=True)

print("Installing demucs...")
pip("demucs")

print("Installing soundfile...")
pip("soundfile")

print("Installing omnizart dependencies...")
pip("madmom --no-build-isolation")

print("Cloning and patching omnizart (vamp-free install)...")
subprocess.run(["git", "clone", "--depth=1",
                "https://github.com/Music-and-Culture-Technology-Lab/omnizart.git",
                "/content/_omnizart_src"], check=True, capture_output=True)

# setup.py から vamp 依存を除去
import re
setup_path = "/content/_omnizart_src/setup.py"
with open(setup_path, "r") as f:
    src = f.read()
src = re.sub(r"['\"]vamp[^'\"]*['\"],?\s*", "", src)
with open(setup_path, "w") as f:
    f.write(src)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "/content/_omnizart_src", "--no-build-isolation"], check=True)

print("✅ インストール完了")

Installing demucs...
Installing soundfile...
Installing omnizart dependencies...
Cloning and patching omnizart (vamp-free install)...


CalledProcessError: Command '['git', 'clone', '--depth=1', 'https://github.com/Music-and-Culture-Technology-Lab/omnizart.git', '/content/_omnizart_src']' returned non-zero exit status 128.

## 3. Google Drive マウント

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("✅ Google Drive マウント完了")

## 4. プロジェクト読み込み

In [ ]:
import sys
from pathlib import Path

# ── Google Drive 上のプロジェクトルート ─────────────────────────────
# 配置場所を変えた場合はここを修正してください
PROJECT_ROOT = Path("/content/drive/MyDrive/drum-copy")
# ────────────────────────────────────────────────────────────────────

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"プロジェクトが見つかりません: {PROJECT_ROOT}\n"
        "Google Drive の 'マイドライブ/drum-copy/' にプロジェクトを配置してください。"
    )

sys.path.insert(0, str(PROJECT_ROOT))
%cd {PROJECT_ROOT}
print(f"✅ プロジェクトパス: {PROJECT_ROOT}")

## 5. パス設定

入力・出力・一時ディレクトリを設定します。  
**通常はここを変更する必要はありません。**

In [ ]:
from backends.colab import ColabBackend

# ── 設定 ────────────────────────────────────────────────────────────
MODEL      = "htdemucs"    # Demucs モデル（htdemucs / htdemucs_ft / mdx_extra など）
OVERWRITE  = False         # True にすると既存の MIDI を上書き
# ────────────────────────────────────────────────────────────────────

backend = ColabBackend(
    project_root=str(PROJECT_ROOT),
    use_gdrive=False,  # Drive は既にマウント済み
    tmp_dir="/content/tmp/drum-copy",
)
backend.setup()
paths = backend.get_paths()

SUPPORTED = {".wav", ".mp3"}
audio_files = [p for p in sorted(paths.input_dir.iterdir()) if p.suffix.lower() in SUPPORTED]

print(f"入力ディレクトリ : {paths.input_dir}")
print(f"出力ディレクトリ : {paths.output_dir}")
print(f"一時ディレクトリ : {paths.tmp_dir}")
print(f"モデル          : {MODEL}")
print()
print(f"入力ファイル ({len(audio_files)} 件):")
for f in audio_files:
    print(f"  - {f.name}")
if not audio_files:
    print(f"⚠️  {paths.input_dir} に音声ファイルが見つかりません。")

### (オプション) Google Drive を使わず直接アップロードする場合

下のセルのコメントを外して実行すると、PCからファイルをアップロードできます。

In [ ]:
# from google.colab import files as colab_files
# import shutil
#
# uploaded = colab_files.upload()
# for fname in uploaded:
#     dest = paths.input_dir / fname
#     shutil.move(fname, dest)
#     print(f"保存: {dest}")
#
# # アップロード後に audio_files を更新
# audio_files = [p for p in sorted(paths.input_dir.iterdir()) if p.suffix.lower() in SUPPORTED]

## 6. パイプライン実行

- **Step 1** : Demucs でドラムトラックを分離 → `tmp/` に保存  
- **Step 2** : omnizart でドラムを採譜 → `output/` に MIDI 保存

> 1曲あたりの目安: GPU(T4) で 2〜5分、CPU では 20〜60分

In [ ]:
import logging
import shutil

from pipeline.stem_separation.separator import separate
from pipeline.transcription.transcriber import transcribe

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("drum_copy")

if not audio_files:
    raise FileNotFoundError(f"入力ファイルがありません: {paths.input_dir}")

results = []
for audio_path in audio_files:
    midi_out = paths.output_dir / (audio_path.stem + ".mid")

    if midi_out.exists() and not OVERWRITE:
        logger.info("スキップ（既存）: %s", midi_out.name)
        results.append((audio_path.name, str(midi_out), "SKIP"))
        continue

    try:
        logger.info("=== Step 1: Stem 分離 — %s ===", audio_path.name)
        drums_wav = separate(audio_path, paths.tmp_dir, model=MODEL)

        logger.info("=== Step 2: MIDI 変換 — %s ===", drums_wav.name)
        midi_path = transcribe(drums_wav, paths.output_dir)

        if midi_path.stem != audio_path.stem:
            renamed = paths.output_dir / (audio_path.stem + ".mid")
            midi_path.rename(renamed)
            midi_path = renamed

        stem_tmp = paths.tmp_dir / audio_path.stem
        if stem_tmp.exists():
            shutil.rmtree(stem_tmp)

        results.append((audio_path.name, str(midi_path), "OK"))
        logger.info("完了: %s", midi_path)

    except Exception as exc:
        logger.error("失敗 %s: %s", audio_path.name, exc)
        results.append((audio_path.name, "", f"ERROR: {exc}"))

print("\n─── 結果 ─────────────────────────────────────")
for src, dst, status in results:
    icon = "✅" if status in ("OK", "SKIP") else "❌"
    print(f"{icon}  {src}  →  {dst or status}")

## 7. MIDI ダウンロード（オプション）

出力 MIDI は Google Drive の `output/` に自動保存されています。  
Colab から直接ダウンロードしたい場合は下のセルを実行してください。

In [ ]:
from google.colab import files as colab_files

midi_files = list(paths.output_dir.glob("*.mid"))
if midi_files:
    for f in midi_files:
        print(f"ダウンロード中: {f.name}")
        colab_files.download(str(f))
else:
    print("出力 MIDI ファイルが見つかりません。")